# Pinceaux Flipbook Viewer
Interactive slider to scroll through slices across stages:
- Raw
- Edge_Corrected
- MLI_only
- Contours

In [1]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

In [2]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'Inputs').exists() and (candidate / 'Outputs').exists():
            return candidate
    raise FileNotFoundError(
        f'Could not find repo root from {start}. Expected folders: Inputs/ and Outputs/'
    )

base_dir = find_repo_root(Path.cwd())


def parse_z_from_name(name: str):
    m = re.search(r'z_(\d+)_(-?\d+\.\d+)\.png$', name)
    if m:
        return int(m.group(1)), float(m.group(2))
    return None


def sort_key(path: Path):
    parsed = parse_z_from_name(path.name)
    if parsed is not None:
        idx, z = parsed
        return (0, idx, z, path.name)
    return (1, path.name)


def get_stage_files(pinceaux_id: int):
    raw_dir = base_dir / 'Inputs' / 'Raw' / f'pinceaux_{pinceaux_id}'
    edge_dir = base_dir / 'Inputs' / 'Edge_Corrected' / f'pinceaux_{pinceaux_id}'
    mli_dir = base_dir / 'Inputs' / 'MLI_only' / f'pinceaux_{pinceaux_id}'
    contours_dir = base_dir / 'Outputs' / f'pinceaux_{pinceaux_id}' / 'Contours'

    return {
        'Raw': sorted(raw_dir.glob('*.png'), key=sort_key),
        'Edge_Corrected': sorted(edge_dir.glob('*.png'), key=sort_key),
        'MLI_only': sorted(mli_dir.glob('*.png'), key=sort_key),
        'Contours': sorted(contours_dir.glob('*.png'), key=sort_key),
    }


def show_frame(stage_files, idx):
    stages = ['Raw', 'Edge_Corrected', 'MLI_only', 'Contours']
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    for ax, stage in zip(axes.ravel(), stages):
        files = stage_files[stage]
        if not files:
            ax.set_title(f'{stage} (no files)')
            ax.axis('off')
            continue

        use_idx = min(idx, len(files) - 1)
        p = files[use_idx]
        img = Image.open(p)
        ax.imshow(img)
        ax.set_title(f'{stage}\n{p.name}')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [3]:
id_widget = widgets.IntText(value=5, description='Pinceaux ID:')
load_button = widgets.Button(description='Load Stages', button_style='info')
slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description='Slice:')
out = widgets.Output()

state = {'files': None}

def on_load(_):
    with out:
        out.clear_output()
        files = get_stage_files(id_widget.value)
        state['files'] = files
        max_len = max((len(v) for v in files.values()), default=0)
        slider.max = max(0, max_len - 1)
        slider.value = 0
        print('Loaded stage files:')
        for k, v in files.items():
            print(f'  {k}: {len(v)}')
        if max_len > 0:
            show_frame(files, slider.value)

def on_slide(change):
    if change['name'] != 'value':
        return
    if state['files'] is None:
        return
    with out:
        out.clear_output(wait=True)
        show_frame(state['files'], slider.value)

load_button.on_click(on_load)
slider.observe(on_slide, names='value')

display(widgets.VBox([id_widget, load_button, slider, out]))